# ELP capture — preview, and take photos

Split out of `pose/online_camera.ipynb`, which runs the *pose* loop. This one only
opens cameras, shows you what they see, and writes frames to disk when you press the
button. Nothing here imports the estimator, so it works before any calibration exists —
which is the point, since the captures are what the calibration is fitted to.

**The preview runs on a background thread.** A `while` loop in the cell would block the
kernel, and a blocked kernel never runs a button callback, so the button would be dead until
the loop finished. The thread keeps the kernel free; the buttons work while frames arrive.

**These are mono sensors.** The OV9281 has no colour to give — chroma is identically zero in
every pixel — so frames are single-channel and saved that way.

**Ask for a mode that exists.** `elp.open_elp(strict=True)` raises when the driver grants
something other than what was asked, rather than substituting quietly: a silent fallback
changes the intrinsics' scale and every distance measured afterwards is wrong by a factor
nothing downstream can detect. Measured modes are in `elp_camera.json`:

| mode | fps | note |
|---|---|---|
| 1280×800 | 121 | native |
| 1280×720 | 121 | cropped from 800 — slower *and* narrower |
| 1024×768 | 120 | native |
| 640×480 | 210 | native |
| **640×400** | **271** | true 0.5× rescale of 1280×800 — the only mode whose intrinsics scale |
| 320×240 | 422 | native |

In [ ]:
import sys, time, threading
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as W
from IPython.display import display

HERE = Path.cwd()
if HERE.name != "camera":                    # tolerate running from the repo root
    HERE = next(p for p in [HERE / "controller/camera",
                            HERE / "ESP32_PMW/controller/camera"] if p.exists())
sys.path[:0] = [str(HERE)]

import elp
import sources

# Captures land beside the ones already in the repo, one directory per session so a
# run is never mixed with an earlier one.
CAPTURES = HERE.parent / "pose" / "assets" / "captures"

print("modes on file:", ", ".join(f"{m.width}x{m.height}" for m in elp.modes()))
print("captures ->", CAPTURES)

## The preview widget

One helper, used by both blocks below. It takes any source that `elp.open_group` returns —
one camera or a stereo pair — and does not care which, because `elp.as_frames` normalises a
read to `(t, [frame, ...])`.

Three things worth knowing about it:

- **The preview thread never writes files.** It keeps the newest frame in a slot; the button
  copies whatever is in that slot. So a capture is always a frame that was actually
  displayed, and pressing the button cannot stall the preview.
- **The JPEG encode is throttled** to `hz`. Encoding at 271 fps costs more than the capture
  does and would make the display rate the thing you are measuring.
- **Stereo frames are saved with the same index and their measured skew**, not assumed to be
  simultaneous. Two free-running USB cameras are not synchronised, and a pair whose skew you
  did not record is a pair you cannot use for calibration later.

In [ ]:
class Preview:
    """Live preview with a capture button. Works for one camera or a pair."""

    def __init__(self, source, cams, outdir, hz=15.0, quality=85, scale=0.5):
        self.source, self.cams, self.hz, self.quality, self.scale = source, cams, hz, quality, scale
        self.outdir = Path(outdir)
        self.outdir.mkdir(parents=True, exist_ok=True)
        self._latest = None          # (t, [frame, ...]) -- newest read, for the button
        self._stop = threading.Event()
        self._thread = None
        self.n_saved = 0

        self.image = W.Image(format="jpeg")
        self.status = W.HTML("<i>starting…</i>")
        self.btn_shot = W.Button(description="Capture", button_style="primary", icon="camera")
        self.btn_stop = W.Button(description="Stop", button_style="danger", icon="stop")
        self.btn_shot.on_click(lambda _b: self.capture())
        self.btn_stop.on_click(lambda _b: self.stop())
        self.ui = W.VBox([self.image, W.HBox([self.btn_shot, self.btn_stop]), self.status])

    # -- display -------------------------------------------------------------
    def _tile(self, frames):
        """One frame, or two side by side with a divider so they cannot be confused."""
        shown = [f if f.ndim == 2 else cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
        if self.scale != 1.0:
            shown = [cv2.resize(f, None, fx=self.scale, fy=self.scale,
                                interpolation=cv2.INTER_AREA) for f in shown]
        if len(shown) == 1:
            return shown[0]
        gap = np.full((shown[0].shape[0], 4), 255, np.uint8)
        return np.hstack([shown[0], gap, shown[1]])

    def _loop(self):
        period = 1.0 / self.hz
        t_drawn = 0.0
        while not self._stop.is_set():
            item = elp.as_frames(self.source.read())
            if item is None:
                time.sleep(0.005)
                continue
            self._latest = item
            now = time.monotonic()
            if now - t_drawn < period:
                continue
            t_drawn = now
            ok, buf = cv2.imencode(".jpg", self._tile(item[1]),
                                   [cv2.IMWRITE_JPEG_QUALITY, self.quality])
            if ok:
                self.image.value = buf.tobytes()
            self._render_status()

    def _render_status(self):
        bits = [f"{c.actual['width']}x{c.actual['height']}" for c in self.cams]
        drops = sum(getattr(c, "n_dropped", 0) for c in self.cams)
        skew = ""
        if len(self.cams) > 1 and hasattr(self.source, "skew_stats"):
            # Already in ms, and cumulative over the session -- see sources.skew_stats.
            st = self.source.skew_stats()
            if st:
                skew = (f" &middot; skew {st['median_ms']:.1f} ms median,"
                        f" {st['max_ms']:.1f} worst")
        self.status.value = (f"{' + '.join(bits)} &middot; saved <b>{self.n_saved}</b>"
                             f" &middot; dropped {drops}{skew}")

    # -- actions -------------------------------------------------------------
    def start(self):
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        display(self.ui)
        return self

    def capture(self):
        item = self._latest
        if item is None:
            self.status.value = "<b>nothing to capture</b> — no frame has arrived yet"
            return
        t, frames = item
        idx = self.n_saved
        names = []
        for c, frame in enumerate(frames):
            p = self.outdir / (f"{idx:03d}.png" if len(frames) == 1 else f"{idx:03d}_cam{c}.png")
            cv2.imwrite(str(p), frame)
            names.append(p.name)
        self.n_saved += 1
        self._render_status()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)
        try:
            self.source.close()
        finally:
            self.btn_shot.disabled = self.btn_stop.disabled = True
            self.status.value = (f"stopped &middot; <b>{self.n_saved}</b> capture(s) in "
                                 f"<code>{self.outdir}</code>")

## 1. One camera

Opens index 0 — on macOS the USB camera enumerates *before* the built-in FaceTime, so the ELP
is `0` and the laptop camera is `1`. `elp.probe_indices()` says what is actually there if that
does not hold on your machine.

Press **Capture** for each shot, **Stop** when finished. Files land as `000.png`, `001.png`, …
in a session directory printed below. PNG, not JPEG: these are measurement inputs, and JPEG
ringing at a black-to-white edge is exactly the signal the segmenter reads.

In [ ]:
INDEX = 0
MODE = "1280x800"

session = CAPTURES / f"elp_{time.strftime('%Y%m%d_%H%M%S')}"
src, cams = elp.open_group(INDEX, mode=MODE)
print(f"camera {INDEX}: asked {MODE}, got {cams[0].actual['width']}x{cams[0].actual['height']}")
print("session ->", session)

mono = Preview(src, cams, session).start()

## 2. Two cameras, simultaneously

The same widget with two indices. `elp.open_group` returns a `StereoSource`, which reads both
and pairs them; `as_frames` hands back `[left, right]` and the preview lays them side by side.

**Watch the skew.** The two cameras free-run — there is no hardware sync — so `StereoSource`
*measures* how far apart each pair actually landed rather than assuming they are simultaneous.
At 121 fps one frame of skew is 8.3 ms. That is fine for a stationary calibration target and
not fine for a moving robot, so it is recorded with every capture and printed at the end.

Each shot writes `000_cam0.png` and `000_cam1.png` — same index, so a pair stays a pair.

If only one camera is connected this raises rather than silently capturing a mono session,
because a stereo directory with half its frames missing is worse than no directory.

In [ ]:
INDICES = [0, 1]
MODE_STEREO = "1280x800"
MAX_SKEW_S = 0.010          # pairs further apart than this are rejected by StereoSource

found = elp.probe_indices(max_index=4)
print("cameras responding at:", found)
if len([i for i in INDICES if i in found]) < 2:
    raise SystemExit(f"need both of {INDICES}; only {found} responded")

session_s = CAPTURES / f"stereo_{time.strftime('%Y%m%d_%H%M%S')}"
pair, pair_cams = elp.open_group(INDICES, mode=MODE_STEREO, max_skew_s=MAX_SKEW_S)
for i, c in zip(INDICES, pair_cams):
    print(f"camera {i}: {c.actual['width']}x{c.actual['height']}")
print("session ->", session_s)

stereo = Preview(pair, pair_cams, session_s).start()

## 3. What you captured

Run after **Stop**. The skew summary is the number that decides whether the pairs are usable:
a median well inside one frame period is a good session, and a long tail means one camera was
starving — usually USB bandwidth, which is fixed by putting the two cameras on separate buses
rather than by anything in software.

In [ ]:
for name, pv in (("mono", globals().get("mono")), ("stereo", globals().get("stereo"))):
    if pv is None:
        continue
    files = sorted(pv.outdir.glob("*.png"))
    print(f"{name}: {len(files)} file(s) in {pv.outdir}")
    st = pv.source.skew_stats() if hasattr(pv.source, "skew_stats") else {}
    if st:
        print(f"   skew over {st['n']} pairs: median {st['median_ms']:.2f} ms, "
              f"p95 {st['p95_ms']:.2f}, worst {st['max_ms']:.2f}   "
              f"(one frame at 121 fps = 8.3 ms; {st['dropped']} pairs rejected)")
    if files:
        f = cv2.imread(str(files[-1]), cv2.IMREAD_UNCHANGED)
        print(f"   last: {files[-1].name}  {f.shape}  {f.dtype}  "
              f"levels p5 {np.percentile(f, 5):.0f} / median {np.median(f):.0f} / p95 {np.percentile(f, 95):.0f}")